In [1]:
import torch

In [5]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [10]:
import os


In [13]:
path=r"D:\Ml Dl\Project\change-detection\artifacts\data_ingestion\LEVIR CD"
subpath=os.listdir(os.path.join(path,"train"))

In [14]:
subpath

['A', 'B', 'label']

In [23]:
subpath=os.path.join(path,"train")
a_path=os.listdir((os.path.join(subpath,"A")))
b_path=os.listdir((os.path.join(subpath,"B")))
label_path=os.listdir((os.path.join(subpath,"label")))

In [24]:
a_path[:5]

['train_1.png',
 'train_10.png',
 'train_100.png',
 'train_101.png',
 'train_102.png']

In [25]:
b_path[:5]

['train_1.png',
 'train_10.png',
 'train_100.png',
 'train_101.png',
 'train_102.png']

In [26]:
label_path[:5]

['train_1.png',
 'train_10.png',
 'train_100.png',
 'train_101.png',
 'train_102.png']

In [65]:
with Image.open(r"D:\Ml Dl\Project\change-detection\artifacts\data_ingestion\LEVIR CD\test\A\test_1.png") as img:
    img = img.convert('L')    
    before_image=np.array(img)

In [66]:

before_image=torch.from_numpy(before_image).float().unsqueeze(0)

before_image.shape

torch.Size([1, 1024, 1024])

In [68]:
before_image.shape[1:]

torch.Size([1024, 1024])

In [48]:
new_H=0
new_W=0
before_image=before_image[new_H:new_H+255,new_W:new_W+255]

In [49]:
before_image.shape

(255, 255, 3)

In [82]:
from torch.utils.data import Dataset
import numpy as np
import PIL
import random
class ImageDataset(Dataset):
    def __init__(self,path,patch_size=256,augmentation=True):
        self.sub_a_path=os.path.join(path,"A")
        self.sub_b_path=os.path.join(path,"B")
        self.sub_label_path=os.path.join(path,"label")
        
        self.a_path=os.listdir(self.sub_a_path)
        self.b_path=os.listdir(self.sub_b_path)
        self.label_path=os.listdir(self.sub_label_path)
        self.patch_size=patch_size
        self.augmentation=augmentation

    def __len__(self):
        return len(self.a_path)

    def __getitem__(self, idx):
        after_im_path=os.path.join(self.sub_a_path,self.a_path[idx])
        before_im_path=os.path.join(self.sub_b_path,self.a_path[idx])
        label_im_path=os.path.join(self.sub_label_path,self.a_path[idx])
        with Image.open(before_im_path) as img:
            img = img.convert('L')
            before_image=np.array(img)
        with Image.open(after_im_path) as img:
            img = img.convert('L')
            after_image=np.array(img)
        with Image.open(label_im_path) as img:
            img = img.convert('L')
            label_image=np.array(img)

        H, W = before_image.shape[:2]

        if self.augmentation:
            new_H=random.randint(0, H - self.patch_size)
            new_W=random.randint(0, W - self.patch_size)

        else :
            new_H=0
            new_W=0

        before_image=before_image[new_H:new_H+self.patch_size,new_W:new_W+self.patch_size]
        after_image=after_image[new_H:new_H+self.patch_size,new_W:new_W+self.patch_size]
        label_image=label_image[new_H:new_H+self.patch_size,new_W:new_W+self.patch_size]

        before_image = torch.from_numpy(before_image).float().unsqueeze(0) / 255.0
        after_image = torch.from_numpy(after_image).float().unsqueeze(0) / 255.0
        
        label_image =torch.from_numpy(label_image).long()
        return before_image,after_image,label_image

In [83]:
from torch.utils.data import DataLoader
path=r"D:\Ml Dl\Project\change-detection\artifacts\data_ingestion\LEVIR CD"
subpath=os.path.join(path,"train")
dataset = ImageDataset(
    path=subpath,
    patch_size=256,
    augmentation=True
)

loader = DataLoader(dataset, batch_size=8, shuffle=True)

In [88]:
a,b,l = next(iter(loader))

In [92]:
a.shape

torch.Size([8, 1, 256, 256])

In [93]:
l.shape

torch.Size([8, 256, 256])